# NóiLái — Kaggle TPU v5e-8 runs (vLLM-TPU)

The TPU variant of the T4 notebook, same structure (the plan a session runs is `PLAN_PATH`,
the pre-registered `configs/run_plan.yaml` by default): the three large models (`tpu_main`:
Gemma 3 12B, Qwen3.8-27B, Qwen-SEA-LION-v4.5-27B-IT in bf16, tensor parallel over the 8
chips), the bf16 drift check (`bf16_drift_200`: 200 core items per T4 model, unquantized;
`phogpt-4b-chat--bf16` is configured for the Modal L4 and skipped here by the session guard)
and the 27B reasoning sub-study (`reasoning_500_tpu`). Sessions last 9 h (~20 TPU-h/week),
one at a time; the vLLM-TPU startup takes 6–22 minutes per model, so keep `MODELS` short
per session and rely on `--resume` (cell (b') restores the previous session's runs tree from
the `noilai-runs` dataset first); outputs are pushed every `PUSH_EVERY_MINUTES` and after every
model. Attach: accelerator **TPU VM v5e-8**, internet on, the same secrets and private
datasets as the T4 notebook (including `noilai-runs` from the second session on).

Every TPU-hosted model must load under vLLM-TPU and return prompt log-probabilities before its
full run (plan §3): the smoke cell checks that.

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/BEEEEEEBAGON/noilai.git"   # the split-out repository (docs/MIGRATION.md); BUNDLE_PATH wins when set
REPO_REF = "main"
REPO_SUBDIR = "noilai"
BUNDLE_PATH = "/kaggle/input/noilai-bundle/noilai-main.bundle"
CLONE_DIR = "/kaggle/working/repo"
WORK_DIR = "/kaggle/working"
HF_HOME = "/kaggle/tmp/hf"                          # 27B bf16 weights are ~54 GB: not in the 20 GB working dir [UNCERTAIN: verify]
ITEMS_DATASET_DIR = "/kaggle/input/noilai-release"
RUNS_RESTORE_DIR = "/kaggle/input/noilai-runs"      # the private runs dataset attached as input; restored by (b') so --resume continues
VERIFY_ITEM_KEYS = None          # None = derived from RUN_IDS and SMOKE_RUN_IDS by (d)
PLAN_PATH = "configs/run_plan.yaml"   # the plan this session executes (its TPU lines are listed under RUN_IDS); "configs/run_plan_exploratory.yaml" holds DEV-ONLY exploratory lines, none of them a TPU line today

VLLM_TPU_PACKAGE = "vllm-tpu"    # [UNCERTAIN: verify] the TPU build's package name for the pinned version (kaggle-tpu-lab documents the working recipe)
VLLM_TPU_VERSION = None          # [UNCERTAIN: verify] pin once the smoke cell passes; None = latest (recorded by (c))

GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"

RUN_IDS = ["tpu_main"]           # lines of PLAN_PATH; any of: tpu_main, bf16_drift_200, reasoning_500_tpu
MODELS = ["gemma-3-12b-it"]      # subset of each run's models; [] = every model the TPU session can run
SMOKE_RUN_IDS = ["smoke_20_tpu"] # (e) the TPU smoke line on SMOKE_MODEL
SMOKE_MODEL = "gemma-3-12b-it"
EXTRA_ARGS = ""
ACCOUNT_HOLDER_ROLE = ""         # role only, never a name (DD 11.2); recorded in every manifest
ALLOW_UNPINNED_REVISION = False  # DD 7.1
PUSH_EVERY_MINUTES = 30
DRY_RUN = False

PLATFORM, GPU_TYPE, N_GPUS = "kaggle", "tpu-v5e-8", 1
SESSION_HARDWARE = "tpu"         # entries configured for t4 / 2xt4 / l4 are skipped here
ALLOW_HARDWARE_MISMATCH = False
MAX_SESSION_HOURS = 8.0          # 9-h TPU sessions: launch no new model past 8 h
REQUIRE_TPU = True
RUN_LABEL = "+".join(RUN_IDS)
PUSH_DATASET = True
PUSH_AFTER_EACH_MODEL = True
KAGGLE_DATASET_SLUG = "noilai-runs"
CREATE_DATASET = False

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# (a) environment check: TPU v5e-8 through JAX / tpu-info; the vLLM-TPU startup takes 6-22 minutes on top
import json, shutil, subprocess, sys
print(sys.version.split()[0], "on", sys.platform)
ENV_CHECK = {"tpu_devices": 0, "device_kind": None}
try:
    import jax
    devs = jax.devices()
    ENV_CHECK["tpu_devices"] = len(devs)
    ENV_CHECK["device_kind"] = getattr(devs[0], "device_kind", str(devs[0])) if devs else None
    print("jax", jax.__version__, "devices:", [str(d) for d in devs])
except Exception as e:  # jax missing or no TPU runtime
    print("jax/TPU not visible from here:", repr(e))
if shutil.which("tpu-info"):
    print(subprocess.run(["tpu-info"], capture_output=True, text=True).stdout)
if REQUIRE_TPU and ENV_CHECK["tpu_devices"] == 0:
    raise SystemExit("no TPU device: choose 'TPU VM v5e-8' in the Kaggle accelerator menu and restart")
du = shutil.disk_usage(WORK_DIR)
print(f"disk at {WORK_DIR}: {du.free / 2**30:.1f} GB free of {du.total / 2**30:.1f} GB")
print(json.dumps(ENV_CHECK))

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
askpass = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
# the environment git runs in is built AFTER the export, so the askpass helper finds the token in it
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if askpass is not None:
    env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
# a persisted clone may hold a STALE local branch named like REPO_REF: try the freshly fetched
# remote-tracking ref first and the literal ref last (commit hashes and tags have no remote form)
import re
cands = [REPO_REF] if re.fullmatch(r"[0-9a-f]{7,40}", REPO_REF) else [f"origin/{REPO_REF}", REPO_REF]
for cand in cands:
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF} (tried {cands})")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (b') restore: the clone is ephemeral, so the runs tree persisted by the previous session (the private
# Kaggle dataset `noilai-runs` attached as input, or the Drive folder the API notebook copies to) is
# copied back into PROJECT/data/runs BEFORE anything runs: outputs.jsonl / manifest.json / unchanged.jsonl
# per run directory (so `--resume` continues at the first undone row instead of item 0), api_ledger.json
# (so a day's quota is not re-spent on rows already answered; DD 7.3 "a hit never re-calls") and the
# compute log (so the session's rows append to the campaign's). Nothing here overwrites a newer local file.
import shutil
from pathlib import Path
RESTORED = {"from": None, "run_dirs": 0, "ledger": False, "compute_log": False}
_src = Path(RUNS_RESTORE_DIR) if RUNS_RESTORE_DIR else None
if _src is not None and _src.exists():
    runs_src = _src / "runs" if (_src / "runs").is_dir() else _src        # the dataset holds runs/ + compute_log.csv, or the runs tree itself
    runs_dst = PROJECT / "data" / "runs"
    runs_dst.mkdir(parents=True, exist_ok=True)
    for child in sorted(runs_src.iterdir()):
        if child.name in ("__pycache__", ".ipynb_checkpoints"):
            continue
        if child.is_dir():
            shutil.copytree(child, runs_dst / child.name, dirs_exist_ok=True)
            RESTORED["run_dirs"] += 1
        elif child.name == "api_ledger.json":
            shutil.copy2(child, runs_dst / child.name)
            RESTORED["ledger"] = True
        elif child.suffix in (".json", ".jsonl", ".txt", ".csv"):
            shutil.copy2(child, runs_dst / child.name)
    for cand in (_src / "compute_log.csv", runs_src / "compute_log.csv"):
        if cand.exists() and not (PROJECT / "data" / "compute_log.csv").exists():
            shutil.copy2(cand, PROJECT / "data" / "compute_log.csv")
            RESTORED["compute_log"] = True
            break
    RESTORED["from"] = str(runs_src)
    print("restored", RESTORED)
else:
    print("nothing to restore (first session, or RUNS_RESTORE_DIR", RUNS_RESTORE_DIR, "is not attached): every run starts at item 0")

In [ ]:
# (c) pinned install for the TPU: the vLLM TPU build first, then the project with its eval extra;
# record pip freeze + versions next to the outputs (checklist C4).
import subprocess, sys
from pathlib import Path
spec = f"{VLLM_TPU_PACKAGE}=={VLLM_TPU_VERSION}" if VLLM_TPU_VERSION else VLLM_TPU_PACKAGE
subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]"], check=True)
RUN_PYTHON = sys.executable          # the TPU image's interpreter carries the vLLM-TPU build [UNCERTAIN: verify]
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
_export(HF_TOKEN_SECRET)
if HF_TOKEN_SECRET != "HF_TOKEN" and os.environ.get(HF_TOKEN_SECRET):
    os.environ["HF_TOKEN"] = os.environ[HF_TOKEN_SECRET]
import colab_setup as CS
ENV_RECORD = CS.record_environment(PROJECT / "data" / "runs" / "env")
print("environment recorded at", ENV_RECORD)

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s) of the runs this
# notebook will execute: the keys are DERIVED from the run ids (plus the smoke run) unless
# VERIFY_ITEM_KEYS lists them; a seeded sub-sample the plan derives from another file is written first
# (--materialize). Each file is checked for its hash against the plan (PLAN_PATH), the header record,
# the canary on every row and its row/cell counts. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
import kaggle_run_plan as KRP
PLAN = KRP.load_plan(Path(PLAN_PATH))                 # the plan this session executes (parameters cell); the run cells load it again
print("plan:", PLAN_PATH, "with runs", [r["id"] for r in PLAN["runs"]])
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    rel = Path(PLAN["release"])                       # e.g. data/release/v0.3: the plan's `release` key decides where the files go
    dest = PROJECT / rel
    if (src / rel.name).exists():                      # the dataset holds the data/release tree (one directory per version)
        shutil.copytree(src / rel.name, dest, dirs_exist_ok=True)
    elif (src / "manifest.json").exists():             # the dataset holds ONE release directory
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        raise SystemExit(f"{src} holds neither {rel.name}/ nor a release manifest.json: attach the frozen release")
    print("release copied from", src, "to", dest)
_run_ids = list(globals().get("RUN_IDS") or [globals().get("RUN_ID")]) + list(globals().get("SMOKE_RUN_IDS") or [])
KEYS = VERIFY_ITEM_KEYS or KRP.item_keys_for_runs(PLAN, [r for r in _run_ids if r])
print("item files to verify:", KEYS)
for key in KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--plan", PLAN_PATH, "--key", key, "--materialize"], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

In [ ]:
# outputs hand-off used after EVERY model (DD 7.6: Kaggle keeps /kaggle/working only if the version
# completes, so a session that times out must already have pushed what it finished) and once more at
# the end: copy data/runs (+ compute log) into /kaggle/working and add a version to a private Kaggle
# dataset through scripts/kaggle_dataset.py (credentials from Secrets, never printed).
import os, shutil, subprocess, sys
from pathlib import Path

def push_outputs(label):
    out = Path(WORK_DIR) / "noilai_runs_out"
    shutil.copytree(PROJECT / "data" / "runs", out / "runs", dirs_exist_ok=True)
    if (PROJECT / "data" / "compute_log.csv").exists():
        shutil.copy2(PROJECT / "data" / "compute_log.csv", out / "compute_log.csv")
    print("outputs copied to", out)
    if not PUSH_DATASET:
        return
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        _export(name)
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        cmd = [sys.executable, "scripts/kaggle_dataset.py", "--src", str(PROJECT / "data" / "runs"),
               "--src", str(PROJECT / "data" / "compute_log.csv"), "--dest", str(Path(WORK_DIR) / "noilai_runs_dataset"),
               "--slug", KAGGLE_DATASET_SLUG, "--message", f"{label} @ {HEAD[:8]}"]
        if CREATE_DATASET and not globals().get("_DATASET_CREATED"):
            cmd.append("--create")
        rc = subprocess.run(cmd, check=False).returncode
        if rc == 0:
            globals()["_DATASET_CREATED"] = True
    else:
        print("Kaggle credentials not found: dataset push skipped (outputs are still under", out, ")")

def after_each_model(res):
    if PUSH_AFTER_EACH_MODEL:
        push_outputs(f"{res['run']}/{res['model']} {res['status']}")

### RUN CELL (e) — smoke test on the TPU
`smoke_20_tpu` on `SMOKE_MODEL` (booked under TPU hours): the model must load under vLLM-TPU
with tensor parallel 8, generate, and return prompt log-probabilities (the T3 forced-choice
scoring needs them).

In [ ]:
# RUN CELL (e): smoke test (smoke_20_tpu) on the TPU model.
import shlex
from pathlib import Path
import kaggle_run_plan as KRP
PLAN = KRP.load_plan(Path(PLAN_PATH))      # the plan of this session (parameters cell)
RUN_EXTRA = shlex.split(EXTRA_ARGS) + (["--account-holder", ACCOUNT_HOLDER_ROLE] if ACCOUNT_HOLDER_ROLE else [])   # role only (DD 11.2)
if not ACCOUNT_HOLDER_ROLE:
    print("WARNING: ACCOUNT_HOLDER_ROLE is empty; the manifests will record account_holder 'unknown' (DD 11.2 asks for the role)")
SMOKE_RESULTS = []
for smoke_id in SMOKE_RUN_IDS:
    SMOKE_RESULTS += KRP.execute(smoke_id, models=[SMOKE_MODEL], platform=PLATFORM, dry_run=DRY_RUN,
                                 continue_on_error=False, extra=RUN_EXTRA, plan=PLAN, python=RUN_PYTHON,
                                 session_hardware=SESSION_HARDWARE, allow_hardware_mismatch=ALLOW_HARDWARE_MISMATCH,
                                 session_t0=SESSION_T0, max_session_hours=MAX_SESSION_HOURS)
for r in SMOKE_RESULTS:
    print(r["run"], r["model"], "->", r["status"])
if any(str(r["status"]).startswith(("failed", "skipped", "refused")) for r in SMOKE_RESULTS):
    raise SystemExit("smoke test failed or was skipped: the TPU route for this model is not ready (fallback: 2xT4 4-bit, labelled quantized)")

### RUN CELL (f) — the TPU runs
One `run_eval.py --resume` command per (run id, model) that the TPU session can run; wall
time per model is logged as TPU device-hours; outputs are pushed after every model; no new
model starts past `MAX_SESSION_HOURS`.

In [ ]:
# RUN CELL (f): each run id in RUN_IDS over MODELS (or all of the run's TPU-runnable models), --resume;
# the outputs are pushed every PUSH_EVERY_MINUTES while a model runs (see the T4 notebook's run cell).
import json, threading
from pathlib import Path
import kaggle_run_plan as KRP
PLAN = KRP.load_plan(Path(PLAN_PATH))      # loaded again here so that an edited PLAN_PATH takes effect without re-running (e)
_stop_push = threading.Event()

def _periodic_push():
    while not _stop_push.wait(PUSH_EVERY_MINUTES * 60):
        try:
            push_outputs("periodic")
        except Exception as e:
            print("periodic push failed:", repr(e))
_pusher = threading.Thread(target=_periodic_push, daemon=True)
_pusher.start()
RESULTS = []
try:
    for run_id in RUN_IDS:
        RESULTS += KRP.execute(run_id, models=MODELS or None, platform=PLATFORM, dry_run=DRY_RUN, continue_on_error=True,
                               extra=RUN_EXTRA, plan=PLAN, python=RUN_PYTHON, session_hardware=SESSION_HARDWARE,
                               allow_hardware_mismatch=ALLOW_HARDWARE_MISMATCH, session_t0=SESSION_T0,
                               max_session_hours=MAX_SESSION_HOURS, after_each=after_each_model,
                               allow_unpinned_revision=ALLOW_UNPINNED_REVISION)
finally:
    _stop_push.set()
    _pusher.join(timeout=5)
print(json.dumps([{"run": r["run"], "model": r["model"], "status": r["status"], "hours": r.get("hours"),
                   "logged_device": r.get("logged_device")} for r in RESULTS], indent=1))

In [ ]:
# (g) final hand-off: the same push as after each model, once more for the session as a whole.
push_outputs(RUN_LABEL)

In [ ]:
# (h) GPU-hours log (checklist C1). kaggle_run_plan wrote one row per executed model run; this adds the
# session's remaining wall time (startup, install, model downloads) as "overhead" and prints the totals.
import shutil
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
run_hours = sum((r.get("hours") or 0.0) for r in (globals().get("RESULTS", []) + globals().get("SMOKE_RESULTS", [])))
overhead = max(0.0, CL.hours_since(SESSION_T0) - run_hours)
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=overhead, run_id=f"session-{RUN_LABEL}", purpose="overhead"))
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
out = Path(WORK_DIR) / "noilai_runs_out"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOG, out / "compute_log.csv")
print("log copied to", out / "compute_log.csv", "-> merge into the repository's data/compute_log.csv")